In [1]:
from datasets import load_dataset

csv_path = "/Users/Samuel/Desktop/SAMUEL MacBook/Samuel Yale/Courses/Thesis/Testing/testgraph2.csv"

dataset = load_dataset(
        "csv", data_files=[csv_path], split="train", delimiter=",", column_names=["title", "text"]
    )

from transformers import (
    DPRContextEncoder,
    DPRContextEncoderTokenizer,
)

dpr_ctx_encoder_model_name = "facebook/dpr-ctx_encoder-multiset-base"

dpr_tokenizer = DPRContextEncoderTokenizer.from_pretrained(dpr_ctx_encoder_model_name)
dpr_encoder = DPRContextEncoder.from_pretrained(dpr_ctx_encoder_model_name)

import numpy as np

def testembed(docs: dict, encoder: DPRContextEncoder = dpr_encoder, tokenizer: DPRContextEncoderTokenizer = dpr_tokenizer):
    input_ids = tokenizer(docs['title'], docs['text'], return_tensors='pt')['input_ids']
    embeddings = encoder(input_ids, return_dict = True).pooler_output
    return {'embeddings': embeddings[0]}

from functools import partial

embedded = dataset.map(partial(testembed))

import faiss

index = faiss.IndexHNSWFlat(768, 128, faiss.METRIC_INNER_PRODUCT)
embedded.add_faiss_index("embeddings", custom_index=index)

import torch
device = torch.device('cpu')

from transformers import AutoTokenizer, RagRetriever, RagSequenceForGeneration

tokenizer = AutoTokenizer.from_pretrained("facebook/rag-sequence-nq")
retriever = RagRetriever.from_pretrained("facebook/rag-sequence-nq", index_name='custom', indexed_dataset=embedded)
model = RagSequenceForGeneration.from_pretrained("facebook/rag-token-nq", retriever=retriever).to(device)

/Users/Samuel/Desktop/SAMUEL MacBook/Samuel Yale/Courses/Thesis/senior-thesis-2/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'DPRQuestionEncoderTokenizer'. 
The class this function is called from is 'DPRContextEncoderTokenizer'.
Some weights of the model checkpoint at facebook/dpr-ctx_encoder-multiset-base were not used when initializing DPRContextEncoder: ['ctx_encoder.bert_model.pooler.dense.bias', 'ctx_encoder.bert_model.pooler.dense.weight']
- This IS expected if you are initializing DPRContextEncoder from the checkpoint of a model trained on another task or with another architecture (e.g. 

In [2]:
input_dict = tokenizer.prepare_seq2seq_batch("how many countries are in europe", return_tensors="pt") 

/Users/Samuel/Desktop/SAMUEL MacBook/Samuel Yale/Courses/Thesis/senior-thesis-2/lib/python3.13/site-packages/transformers/models/rag/tokenization_rag.py:88: FutureWarning: `prepare_seq2seq_batch` is deprecated and will be removed in version 5 of 🤗 Transformers. Use the regular `__call__` method to prepare your inputs and the tokenizer under the `with_target_tokenizer` context manager to prepare your targets. See the documentation of your specific tokenizer for more details
  warnings.warn(


In [3]:
print(input_dict['input_ids'])

tensor([[ 101, 2129, 2116, 3032, 2024, 1999, 2885,  102]])


In [4]:
generated = model.generate(input_ids=input_dict["input_ids"])

: 

In [1]:
from datasets import load_dataset

csv_path = "/Users/Samuel/Desktop/SAMUEL MacBook/Samuel Yale/Courses/Thesis/Testing/testgraph2.csv"

dataset = load_dataset(
        "csv", data_files=[csv_path], split="train", delimiter=",", column_names=["title", "text"]
    )

/Users/Samuel/Desktop/SAMUEL MacBook/Samuel Yale/Courses/Thesis/senior-thesis-2/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [9]:
import torch
torch.set_grad_enabled(False)
if torch.cuda.is_available():
    device = torch.device("cuda")
    print('CUDA')
elif torch.backends.mps.is_available():
    if torch.backends.mps.is_built():
        device =torch.device('mps')
        print('MPS')
else:
    device = torch.device("cpu")
    print('CPU')

MPS


In [2]:
from transformers import (
    DPRContextEncoder,
    DPRContextEncoderTokenizer,
)

dpr_ctx_encoder_model_name = "facebook/dpr-ctx_encoder-multiset-base"

dpr_tokenizer = DPRContextEncoderTokenizer.from_pretrained(dpr_ctx_encoder_model_name)
dpr_encoder = DPRContextEncoder.from_pretrained(dpr_ctx_encoder_model_name)

The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'DPRQuestionEncoderTokenizer'. 
The class this function is called from is 'DPRContextEncoderTokenizer'.
Some weights of the model checkpoint at facebook/dpr-ctx_encoder-multiset-base were not used when initializing DPRContextEncoder: ['ctx_encoder.bert_model.pooler.dense.bias', 'ctx_encoder.bert_model.pooler.dense.weight']
- This IS expected if you are initializing DPRContextEncoder from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing DPRContextEncoder from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification mode

In [3]:
import numpy as np

def testembed(docs: dict, encoder: DPRContextEncoder = dpr_encoder, tokenizer: DPRContextEncoderTokenizer = dpr_tokenizer):
    input_ids = tokenizer(docs['title'], docs['text'], return_tensors='pt')['input_ids']
    embeddings = encoder(input_ids, return_dict = True).pooler_output
    return {'embeddings': embeddings[0]}

In [4]:
from functools import partial

embedded = dataset.map(partial(testembed))

In [5]:
import os

context_dir = '/Users/Samuel/Desktop/SAMUEL MacBook/Samuel Yale/Courses/Thesis/Datasets/tests/contexts'
ver = 1
passages_path = os.path.join(context_dir, f"my_knowledge_dataset_{ver}")

embedded.save_to_disk(passages_path)

Saving the dataset (1/1 shards): 100%|██████████| 2/2 [00:00<00:00, 464.59 examples/s]


In [98]:
# from datasets import load_from_disk
# dataset = load_from_disk(passages_path)

In [6]:
import faiss

index = faiss.IndexHNSWFlat(768, 128, faiss.METRIC_INNER_PRODUCT)
embedded.add_faiss_index("embeddings", custom_index=index)

100%|██████████| 1/1 [00:00<00:00, 401.75it/s]


Dataset({
    features: ['title', 'text', 'embeddings'],
    num_rows: 2
})

In [7]:
index_dir = '/Users/Samuel/Desktop/SAMUEL MacBook/Samuel Yale/Courses/Thesis/Datasets/tests/indices'
index_path = os.path.join(index_dir, f"my_knowledge_dataset_{ver}_hnsw_index.faiss")

embedded.get_index("embeddings").save(index_path)

In [101]:
# dataset.load_faiss_index("embeddings", index_path)

In [10]:
device = torch.device('cpu')

In [11]:
from transformers import AutoTokenizer, RagRetriever, RagSequenceForGeneration

tokenizer = AutoTokenizer.from_pretrained("facebook/rag-sequence-nq")
retriever = RagRetriever.from_pretrained("facebook/rag-sequence-nq", index_name='custom', indexed_dataset=embedded)
model = RagSequenceForGeneration.from_pretrained("facebook/rag-token-nq", retriever=retriever).to(device)

The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'RagTokenizer'. 
The class this function is called from is 'DPRQuestionEncoderTokenizer'.
The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'RagTokenizer'. 
The class this function is called from is 'DPRQuestionEncoderTokenizerFast'.
The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'RagTokenizer'. 
The class this function is called from is 'BartTokenizer'.
The tokenizer class you load from this checkpoint is not the same type as the class this function is called fr

In [12]:
device = torch.device('cpu')

In [13]:
inputs = tokenizer("Who is Jack’s son?", return_tensors="pt").to(device)
input_ids = inputs["input_ids"]
outputs = model(input_ids=input_ids)

In [ ]:
inputs = tokenizer("Who is Jack’s son?", return_tensors="pt").to(device)
targets = tokenizer(text_target="In Paris, there are 10 million people.", return_tensors="pt")
input_ids = inputs["input_ids"]
labels = targets["input_ids"]
# outputs = model(input_ids=input_ids)
# 1. Encode
question_hidden_states = model.question_encoder(input_ids)[0].to(device)
# 2. Retrieve
docs_dict = retriever(input_ids.cpu().numpy(), question_hidden_states.detach().cpu().numpy(), return_tensors="pt")
doc_scores = torch.bmm(
    question_hidden_states.unsqueeze(1), docs_dict["retrieved_doc_embeds"].float().transpose(1, 2)
).squeeze(1)
# 3. Forward to generator
outputs = model(
    context_input_ids=docs_dict["context_input_ids"],
    context_attention_mask=docs_dict["context_attention_mask"],
    doc_scores=doc_scores,
    decoder_input_ids=labels,
)

In [ ]:
question = "Who is Jack’s son?"
input_ids = tokenizer.question_encoder(question, return_tensors='pt')['input_ids'].to(device)
outputs = model.generate(input_ids=input_ids)
out_string = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]
print(out_string)

ValueError: The following `model_kwargs` are not used by the model: ['device'] (note: typos in the generate arguments will also show up in this list)

In [ ]:
from transformers import RagTokenizer, RagRetriever, RagSequenceForGeneration

rag_name = 'facebook/dpr-ctx_encoder-single-nq-base'

retriever = RagRetriever.from_pretrained(
    rag_name, index_name="custom", indexed_dataset=embedded
)
rag_model = RagSequenceForGeneration.from_pretrained(rag_name, retriever=retriever)
rag_tokenizer = RagTokenizer.from_pretrained(rag_name)